# Short-Term vs Long-Term Agent Memory: A 26.6 Lifecycle Walkthrough

This companion notebook implements the lifecycle described in the Oracle Developers article:

1. active thread state and summaries;
2. typed durable memory;
3. metadata and thread scope;
4. context-card assembly;
5. updates and TTL;
6. message, memory, thread, and user deletion.

> **Execution status:** Structurally validated, but not executed in this environment because it requires Oracle AI Database credentials, a compatible embedding route, and an LLM provider key. Run top-to-bottom in the target Oracle environment before publication.

## Goal

Build a small retrieval-debugging assistant and verify that:

- exact-thread retrieval excludes a deliberately conflicting cross-thread record;
- cross-thread retrieval can include it when explicitly allowed;
- a context card combines summary, durable records, and recent messages;
- a corrected preference is updated rather than duplicated;
- TTL and explicit deletion remain separate;
- deleting a raw message does not imply deleting durable memory.

## Setup

### Prerequisites

- Oracle AI Database 26ai or later
- Python 3.10 through 3.13
- `oracleagentmemory==26.6.0`
- Oracle database credentials
- An embedding route
- An LLM key if extraction or model-generated summaries are enabled

The default route below uses an external embedding model and vector search. Set `OAMP_EMBED_BACKEND=indb` to use `OracleDBEmbedder` and hybrid search with a database-resident ONNX model.

In [ ]:
%pip install --upgrade pip
%pip install "oracleagentmemory==26.6.0" oracledb

In [ ]:
import os
import sys

import oracleagentmemory

if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError("Use Python 3.10 through 3.13.")

assert oracleagentmemory.__version__ == "26.6.0", oracleagentmemory.__version__
print("oracleagentmemory:", oracleagentmemory.__version__)
print("Python:", sys.version.split()[0])

Set these environment variables before continuing:

```bash
export DB_USER="..."
export DB_PASSWORD="..."
export DB_CONNECT_STRING="..."
export OPENAI_API_KEY="..."

# Optional in-database route
export OAMP_EMBED_BACKEND="indb"
export INDB_EMBED_MODEL="ALL_MINILM_L12_V2"
export INDB_EMBED_DIM="384"
```

In [ ]:
required_env = ["DB_USER", "DB_PASSWORD", "DB_CONNECT_STRING", "OPENAI_API_KEY"]
missing = [name for name in required_env if not os.environ.get(name)]
if missing:
    raise EnvironmentError(f"Missing required environment variables: {missing}")

## Step 1: Create the store and lifecycle policy

In [ ]:
import oracledb

from oracleagentmemory.core import (
    BackgroundExtractionQueueFullBehavior,
    MemoryExtractionConfig,
    MemoryExtractionMode,
    MemoryRetentionConfig,
    OracleAgentMemory,
    OracleDBMemoryStore,
    SchemaPolicy,
    SearchIndexSyncMode,
    SearchStrategy,
)
from oracleagentmemory.core.embedders import Embedder, OracleDBEmbedder
from oracleagentmemory.core.llms import Llm
from oracleagentmemory.core.retention import TimeToLiveAnchor
from oracleagentmemory.apis.thread import Message

db_pool = oracledb.create_pool(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
    min=1,
    max=8,
    increment=1,
)

memory_llm = Llm(
    os.environ.get("OAMP_LLM_MODEL", "gpt-5.4"),
    reasoning_effort="low",
    max_tokens=2_000,
)

In [ ]:
EMBED_BACKEND = os.environ.get("OAMP_EMBED_BACKEND", "openai").lower()

if EMBED_BACKEND == "openai":
    embed_dim = 1536
    embedder = Embedder(
        model=os.environ.get("OAMP_EMBED_MODEL", "text-embedding-3-small"),
        embedding_dimension=embed_dim,
        max_input_tokens=8_191,
        normalize=True,
    )
    search_strategy = SearchStrategy.VECTOR
elif EMBED_BACKEND == "indb":
    embed_dim = int(os.environ.get("INDB_EMBED_DIM", "384"))
    embedder = OracleDBEmbedder(
        connection=db_pool,
        model=os.environ.get("INDB_EMBED_MODEL", "ALL_MINILM_L12_V2"),
        input_name=os.environ.get("INDB_EMBED_INPUT", "DATA"),
        embedding_dimension=embed_dim,
        max_input_tokens=int(os.environ.get("INDB_EMBED_MAX_TOKENS", "512")),
        normalize=True,
        batch_size=16,
    )
    search_strategy = SearchStrategy.HYBRID
else:
    raise ValueError("OAMP_EMBED_BACKEND must be 'openai' or 'indb'.")

print("Embedding route:", EMBED_BACKEND)
print("Search strategy:", search_strategy.name)

In [ ]:
extraction_config = MemoryExtractionConfig(
    extract_memories=True,
    extraction_mode=MemoryExtractionMode.BACKGROUND,
    background_extraction_queue_full_behavior=(
        BackgroundExtractionQueueFullBehavior.WAIT_THEN_RAISE
    ),
    background_extraction_queue_put_timeout_seconds=10.0,
    memory_extraction_frequency=1,
    memory_extraction_window=-1,
    memory_extraction_token_limit=6_000,
    enable_context_summary=True,
    context_summary_update_frequency=2,
    memory_extraction_custom_instructions=(
        "Keep confirmed project facts, stable preferences, decisions, and "
        "reusable guidelines. Ignore greetings, credentials, speculation, "
        "and temporary debugging output."
    ),
    memory_extraction_inherit_message_metadata=[
        "tenant_id", "project_id", "workflow"
    ],
)

retention_config = MemoryRetentionConfig(
    default_ttl_days=90,
    max_ttl_days=365,
)

store_kwargs = dict(
    embedder=embedder,
    pool=db_pool,
    schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,
    memory_store_id="MEMLIFECYCLE",
    search_strategy=search_strategy,
    memory_retention_config=retention_config,
)

if search_strategy is SearchStrategy.VECTOR:
    store_kwargs["vector_dim"] = embed_dim
else:
    store_kwargs["search_index_sync"] = SearchIndexSyncMode.ON_COMMIT

store = OracleDBMemoryStore(**store_kwargs)
memory = OracleAgentMemory(
    store=store,
    llm=memory_llm,
    memory_extraction_config=extraction_config,
)

print("Memory store ready.")

## Step 2: Create two threads for the same user and agent

In [ ]:
from uuid import uuid4

run_id = uuid4().hex[:8]
tenant_id = "acme"
project_id = "PROJECT-17"
user_id = f"developer-{run_id}"
agent_id = f"retrieval-debugger-{run_id}"
thread_a_id = f"debug-main-{run_id}"
thread_b_id = f"debug-other-{run_id}"

metadata = {
    "tenant_id": tenant_id,
    "project_id": project_id,
    "workflow": "retrieval-debugging",
}

memory.add_user(
    user_id,
    "Developer working on retrieval quality.",
    metadata={"tenant_id": tenant_id},
)
memory.add_agent(
    agent_id,
    "Assistant for debugging memory retrieval.",
    metadata={"tenant_id": tenant_id},
)

thread_a = memory.create_thread(
    thread_id=thread_a_id,
    user_id=user_id,
    agent_id=agent_id,
    metadata=metadata,
    context_card_token_limit=6_000,
    context_card_type_search_concurrency=4,
)
thread_b = memory.create_thread(
    thread_id=thread_b_id,
    user_id=user_id,
    agent_id=agent_id,
    metadata={**metadata, "workflow": "unrelated-ui-debugging"},
)

print("Thread A:", thread_a.thread_id)
print("Thread B:", thread_b.thread_id)

## Step 3: Add active state one turn at a time

Automatic extraction frequency counts `add_messages()` calls, so append turns incrementally rather than flushing a whole transcript in one batch.

In [ ]:
message_ids = []

for message in [
    Message(
        role="user",
        content=(
            "We are debugging broad memory retrieval in PROJECT-17. "
            "The stack is Python and Oracle AI Database 26ai."
        ),
    ),
    Message(
        role="assistant",
        content=(
            "I will inspect user, agent, thread, record-type, and metadata scope "
            "before changing top_k or the embedding model."
        ),
    ),
]:
    ids = await thread_a.add_messages_async([message], metadata=metadata)
    message_ids.extend(ids)

await thread_a.wait_for_memory_extraction_async(timeout=120)

print("Message IDs:", message_ids)

In [ ]:
summary = await thread_a.get_summary_async(
    except_last=1,
    token_budget=300,
)

print(summary.content)
assert summary.content.strip()

## Step 4: Add typed durable memories with metadata and TTL

In [ ]:
preference_id = f"pref-python-{run_id}"
main_fact_id = f"fact-main-{run_id}"
conflicting_fact_id = f"fact-other-{run_id}"

await thread_a.add_memory_async(
    "The developer prefers concise Python examples.",
    memory_type="preference",
    memory_id=preference_id,
    metadata=metadata,
    ttl_days=180,
    ttl_anchor=TimeToLiveAnchor.CREATED_AT,
)

await thread_a.add_memory_async(
    "PROJECT-17 retrieval results are too broad; inspect scope first.",
    memory_type="fact",
    memory_id=main_fact_id,
    metadata=metadata,
    ttl_days=90,
    ttl_anchor=TimeToLiveAnchor.CREATED_AT,
)

await thread_b.add_memory_async(
    "The UI debugging workflow concluded that retrieval scope is already correct.",
    memory_type="fact",
    memory_id=conflicting_fact_id,
    metadata={**metadata, "workflow": "unrelated-ui-debugging"},
    ttl_days=90,
    ttl_anchor=TimeToLiveAnchor.CREATED_AT,
)

print("Preference:", preference_id)
print("Main fact:", main_fact_id)
print("Cross-thread fact:", conflicting_fact_id)

## Step 5: Prove scope before ranking

In [ ]:
exact_thread_hits = await thread_a.search_async(
    "What did we conclude about retrieval scope?",
    exact_user_match=True,
    exact_agent_match=True,
    exact_thread_match=True,
    max_results=10,
    record_types=["fact"],
    metadata_filter={"tenant_id": tenant_id},
)

for hit in exact_thread_hits:
    print(hit.record.id, hit.record.record_type, hit.distance, hit.content)

assert any(hit.record.id == main_fact_id for hit in exact_thread_hits)
assert all(hit.record.id != conflicting_fact_id for hit in exact_thread_hits)
print("Exact-thread search excluded the cross-thread record.")

In [ ]:
cross_thread_hits = await thread_a.search_async(
    "What did we conclude about retrieval scope?",
    exact_user_match=True,
    exact_agent_match=True,
    exact_thread_match=False,
    max_results=10,
    record_types=["fact"],
    metadata_filter={"tenant_id": tenant_id},
)

for hit in cross_thread_hits:
    print(hit.record.id, hit.record.record_type, hit.distance, hit.content)

assert any(hit.record.id == conflicting_fact_id for hit in cross_thread_hits)
print("Cross-thread search included the deliberately conflicting record.")

## Step 6: Build a context card

In [ ]:
context_card = await thread_a.get_context_card_async(
    fallback_message_count=6,
    max_relevant_results=8,
    max_recent_messages=4,
    min_relevant_results_by_type={
        "preference": 1,
        "fact": 1,
    },
)

print(context_card.content)
assert "<context_card" in context_card.content

## Step 7: Correct a preference with an update

In [ ]:
await thread_a.update_memory_async(
    preference_id,
    content="The developer now prefers concise TypeScript examples.",
    metadata={
        **metadata,
        "source": "explicit-correction",
    },
    ttl_days=180,
    ttl_anchor=TimeToLiveAnchor.CREATED_AT,
)

corrected_hits = await thread_a.search_async(
    "preferred programming language for examples",
    exact_user_match=True,
    exact_agent_match=True,
    exact_thread_match=True,
    max_results=10,
    record_types=["preference"],
)

for hit in corrected_hits:
    print(hit.record.id, hit.content)

assert any(
    hit.record.id == preference_id and "TypeScript" in hit.content
    for hit in corrected_hits
)

## Step 8: Demonstrate TTL refresh and explicit deletion

In [ ]:
temporary_id = await thread_a.add_memory_async(
    "Temporary benchmark run is scheduled for tomorrow.",
    memory_type="memory",
    metadata={**metadata, "operational": True},
    ttl_days=7,
    ttl_anchor=TimeToLiveAnchor.CREATED_AT,
)

await thread_a.update_memory_async(
    temporary_id,
    ttl_days=30,
    ttl_anchor=TimeToLiveAnchor.CREATED_AT,
)

deleted_count = thread_a.delete_memory(temporary_id)
print("Deleted temporary memory rows:", deleted_count)
assert deleted_count == 1

## Step 9: Keep message and durable-memory deletion separate

Deleting a raw message does not mean that durable records associated with the conversation disappear. Oracle AI Agent Memory does not use per-message provenance to cascade message deletion into extracted memories.

In [ ]:
message_deleted = thread_a.delete_message(message_ids[0])
print("Raw message deleted:", message_deleted)
assert message_deleted == 1

remaining_preference_hits = await thread_a.search_async(
    "preferred language for examples",
    exact_user_match=True,
    exact_agent_match=True,
    exact_thread_match=True,
    max_results=10,
    record_types=["preference"],
)

assert any(hit.record.id == preference_id for hit in remaining_preference_hits)
print("Durable preference remains after raw-message deletion.")

## Step 10: Cleanup

Before strict thread deletion, stop new writes and wait for background extraction accepted through this client. The SDK cannot serialize concurrent work submitted by every other process.

In [ ]:
await thread_a.wait_for_memory_extraction_async(timeout=120)
await thread_b.wait_for_memory_extraction_async(timeout=120)

deleted_thread_a = memory.delete_thread(thread_a_id)
deleted_thread_b = memory.delete_thread(thread_b_id)
deleted_user = memory.delete_user(user_id, cascade=True)
deleted_agent = memory.delete_agent(agent_id, cascade=True)

print("Thread A deletion:", deleted_thread_a)
print("Thread B deletion:", deleted_thread_b)
print("User deletion:", deleted_user)
print("Agent deletion:", deleted_agent)

memory.close()
db_pool.close()

## Checks completed

After execution, confirm that:

- the installed SDK is exactly 26.6.0;
- exact-thread search excludes the cross-thread fact;
- cross-thread search includes it when allowed;
- the context card contains summary and selected durable context;
- the corrected preference uses the same record ID;
- explicit deletion returns the expected count;
- the durable preference remains after deleting a raw message;
- cleanup completes without pending background work.

## Next challenge

Add a third thread with a different `tenant_id`. Verify that exact ownership and metadata filters exclude it even when the text is more semantically similar than the correct record.